# FlowMorph on FLUX.2 Klein Base 9B with an optional LoRA

This thin notebook delegates all model, optimization, rendering, metrics, and packaging logic to `flowmorph_klein`. The reference run is fixed at 512×512, 100 source steps, 100 target steps, 100 scheduler points, CFG 4.0, and 20 frames. It never substitutes a 4B, distilled, or FLUX.1 model. `/content` is temporary, so complete the export section before the runtime ends.

In [1]:
# Editable plain-Python configuration (widgets are not required).
PROJECT_ROOT = "/content/FlowMorphKlein9B"
INPUT_ROOT = "/content/flowmorph_klein_images/max_v1"
WORK_ROOT = "/content/flowmorph_klein_work/max_v1"
RESULT_ROOT = "/content/flowmorph_klein_results/max_v1/full_lora_reproduction_v1"
HF_CACHE_DIR = "/content/hf_cache"
DRIVE_ROOT = "/content/drive/MyDrive/FlowMorphKlein9B"
CONFIG_PATH = f"{PROJECT_ROOT}/configs/full_9b_lora.yaml"
REPOSITORY_URL = "https://github.com/MNoichl/FluxFlowMorph.git"
UPDATE_REPOSITORY = False

MODEL_ID = "black-forest-labs/FLUX.2-klein-base-9B"
MODEL_REVISION = "32773329fbe7e81a90ef971740e8ba4b0364ecf3"
LORA_SOURCE = None  # org/repo, HF page/resolve URL, or local .safetensors
LORA_REVISION = None
LORA_WEIGHT_NAME = None
LORA_SCALE = 1.0

SOURCE_IMAGE = None  # e.g. f"{INPUT_ROOT}/images/source.png"
TARGET_IMAGE = None  # e.g. f"{INPUT_ROOT}/images/target.png"
SOURCE_PROMPT = None
TARGET_PROMPT = None
BRIDGE_PROMPT = "a smooth transformation between the two subjects"
NEGATIVE_PROMPT = ""

PROFILE = "auto"
SEED = 42
RESOLUTION = 512
GUIDANCE_SCALE = 4.0
OUTPUT_NAME = "full_lora_reproduction_v1"
MOUNT_DRIVE = False
UPLOAD_INPUTS = False
RESUME_IF_AVAILABLE = False
RESUME_RUN_DIRECTORY = None  # Existing RUN_ID directory; required when resuming.
COPY_ARCHIVE_TO_DRIVE = False
DOWNLOAD_ARCHIVE = True

## 1. Runtime identification

In [2]:
import platform
import sys
print({"python": sys.version, "platform": platform.platform(), "executable": sys.executable})

{'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'executable': '/usr/bin/python3'}


## 2. GPU and VRAM diagnostics

In [3]:
try:
    import torch
except ImportError:
    print("PyTorch is not installed yet; dependency installation follows in section 5.")
else:
    if torch.cuda.is_available():
        device = torch.cuda.get_device_properties(0)
        free_bytes, total_bytes = torch.cuda.mem_get_info(0)
        print({"selected_device": "cuda:0", "gpu": device.name, "total_vram_gib": total_bytes / 2**30, "free_vram_gib": free_bytes / 2**30, "bf16": torch.cuda.is_bf16_supported()})
    else:
        print({"selected_device": "cpu", "cuda_available": False, "production_supported": False})

{'selected_device': 'cuda:0', 'gpu': 'NVIDIA RTX PRO 6000 Blackwell Server Edition', 'total_vram_gib': 94.97076416015625, 'free_vram_gib': 94.4256591796875, 'bf16': True}


## 3. Path configuration

In [4]:
from pathlib import Path
for local_root in (INPUT_ROOT, WORK_ROOT, RESULT_ROOT, HF_CACHE_DIR):
    Path(local_root).mkdir(parents=True, exist_ok=True)
print({"project": PROJECT_ROOT, "inputs": INPUT_ROOT, "work": WORK_ROOT, "results": RESULT_ROOT, "hf_cache": HF_CACHE_DIR})

{'project': '/content/FlowMorphKlein9B', 'inputs': '/content/flowmorph_klein_images/max_v1', 'work': '/content/flowmorph_klein_work/max_v1', 'results': '/content/flowmorph_klein_results/max_v1/full_lora_reproduction_v1', 'hf_cache': '/content/hf_cache'}


## 4. Repository clone or update

In [5]:
import subprocess
project_path = Path(PROJECT_ROOT)
if not (project_path / "pyproject.toml").is_file():
    if REPOSITORY_URL is None:
        raise RuntimeError("Stage the repository at PROJECT_ROOT or set REPOSITORY_URL.")
    subprocess.check_call(["git", "clone", "--depth", "1", REPOSITORY_URL, PROJECT_ROOT])
elif UPDATE_REPOSITORY:
    subprocess.check_call(["git", "-C", PROJECT_ROOT, "pull", "--ff-only"])
print("Repository ready:", project_path)

RuntimeError: Stage the repository at PROJECT_ROOT or set REPOSITORY_URL.

## 5. Dependency installation

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", f"{PROJECT_ROOT}/requirements-colab.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PROJECT_ROOT])
print("Pinned dependencies and editable package installed.")

## 6. Hugging Face authentication

In [ ]:
from flowmorph_klein.environment import resolve_hf_token
AUTHENTICATION = resolve_hf_token()
print("Authentication source:", AUTHENTICATION.source)  # Credential value is never displayed.

## 7. Model license and access preflight

In [ ]:
from flowmorph_klein.environment import verify_model_access
MODEL_ACCESS = verify_model_access(AUTHENTICATION, model_id=MODEL_ID, revision=MODEL_REVISION)
print(MODEL_ACCESS)

## 8. Optional Google Drive mount

In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        print("Drive mounting is unavailable outside Google Colab.")
    else:
        drive.mount("/content/drive")
        Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
else:
    print("Drive mounting disabled; the active run remains on local storage.")

## 9. Input upload or staging

In [ ]:
from flowmorph_klein.colab_io import detect_colab, stage_uploaded_files
if UPLOAD_INPUTS:
    if not detect_colab():
        raise RuntimeError("Interactive upload is unavailable; copy files locally and set SOURCE_IMAGE/TARGET_IMAGE.")
    uploaded = stage_uploaded_files(Path(INPUT_ROOT) / "images")
    print("Staged uploads:", [str(item.destination) for item in uploaded])
if SOURCE_IMAGE is None or TARGET_IMAGE is None:
    raise RuntimeError("Set SOURCE_IMAGE and TARGET_IMAGE before continuing.")
if not Path(SOURCE_IMAGE).is_file() or not Path(TARGET_IMAGE).is_file():
    raise FileNotFoundError("Both endpoint image paths must exist before model loading.")
print("Endpoint files validated.")

## 10. Run configuration

In [ ]:
from flowmorph_klein.cli import select_hardware_profile
from flowmorph_klein.config import load_config, resolve_config
overrides = {
    "project.name": OUTPUT_NAME, "model.id": MODEL_ID, "model.revision": MODEL_REVISION,
    "lora.source": LORA_SOURCE, "lora.revision": LORA_REVISION, "lora.weight_name": LORA_WEIGHT_NAME,
    "lora.fit_scale": LORA_SCALE, "lora.render_scale": LORA_SCALE,
    "input.source_image": SOURCE_IMAGE, "input.target_image": TARGET_IMAGE,
    "input.source_prompt": SOURCE_PROMPT, "input.target_prompt": TARGET_PROMPT,
    "input.bridge_prompt": BRIDGE_PROMPT, "input.negative_prompt": NEGATIVE_PROMPT,
    "input.width": RESOLUTION, "input.height": RESOLUTION,
    "guidance.scale": GUIDANCE_SCALE, "reproducibility.seed": SEED,
    "paths.input_root": INPUT_ROOT, "paths.work_root": WORK_ROOT,
    "paths.result_root": RESULT_ROOT, "paths.hf_cache": HF_CACHE_DIR, "paths.drive_root": DRIVE_ROOT,
}
TEMPLATE_CONFIG = load_config(CONFIG_PATH, overrides=overrides)
SELECTED_PROFILE = select_hardware_profile(PROFILE if PROFILE != "auto" else TEMPLATE_CONFIG.model.profile)
CONFIG = resolve_config(TEMPLATE_CONFIG, selected_profile=SELECTED_PROFILE, check_input_files=True)
print(CONFIG.model_dump(mode="json", exclude={"lora": {"source"}}))

## 11. Environment validation

In [ ]:
from flowmorph_klein.environment import collect_environment, require_cuda_for_production
SELECTED_DEVICE = require_cuda_for_production()
ENVIRONMENT = collect_environment()
print({"selected_device": str(SELECTED_DEVICE), "gpu": ENVIRONMENT.get("cuda_devices"), "packages": ENVIRONMENT.get("packages")})

## 12. Model and LoRA loading

In [ ]:
from flowmorph_klein.pipeline import FlowMorphRunner
if RESUME_IF_AVAILABLE and not RESUME_RUN_DIRECTORY:
    raise ValueError("Set RESUME_RUN_DIRECTORY to the existing compatible RUN_ID directory.")
if RESUME_IF_AVAILABLE:
    runner = FlowMorphRunner.from_config(CONFIG, run_directory=RESUME_RUN_DIRECTORY)
else:
    runner = FlowMorphRunner.from_config(CONFIG)
if RESUME_IF_AVAILABLE:
    PREPARE_REPORT = runner.prepare(resume=True)
else:
    PREPARE_REPORT = runner.prepare()
print("Base-9B runner prepared; optional adapter validation is recorded by the package.")

## 13. Production backward probe

In [ ]:
BACKWARD_PROBE = runner.run_production_backward_probe()
print(BACKWARD_PROBE)

## 14. Source endpoint fitting

The package facade owns the required source-save-target-save-render lifecycle. This single call starts that lifecycle so no optimizer or model logic is duplicated in notebook cells.

In [ ]:
RUN_RESULT = runner.resume() if RESUME_IF_AVAILABLE else runner.run(resume=False)
RUN_DIRECTORY = Path(runner.run_directory)
print("Workflow returned; run directory:", RUN_DIRECTORY)

## 15. Source checkpoint confirmation

In [ ]:
SOURCE_CHECKPOINT = RUN_DIRECTORY / "checkpoints/source/tensors.safetensors"
assert SOURCE_CHECKPOINT.is_file(), f"Missing source checkpoint: {SOURCE_CHECKPOINT}"
print(SOURCE_CHECKPOINT)

## 16. Target endpoint fitting

In [ ]:
print("Target fitting ran sequentially after the source checkpoint; see execution.log and target_loss.csv.")

## 17. Target checkpoint confirmation

In [ ]:
TARGET_CHECKPOINT = RUN_DIRECTORY / "checkpoints/target/tensors.safetensors"
assert TARGET_CHECKPOINT.is_file(), f"Missing target checkpoint: {TARGET_CHECKPOINT}"
print(TARGET_CHECKPOINT)

## 18. Morph rendering

In [ ]:
RAW_FRAMES = sorted((RUN_DIRECTORY / "raw_frames").glob("frame_*.png"))
DISPLAY_FRAMES = sorted((RUN_DIRECTORY / "display_frames").glob("frame_*.png"))
assert len(RAW_FRAMES) == len(DISPLAY_FRAMES) == CONFIG.flowmorph.frame_count
print({"raw_frames": len(RAW_FRAMES), "display_frames": len(DISPLAY_FRAMES)})

## 19. Metrics

In [ ]:
import json
METRICS_PATH = RUN_DIRECTORY / "metrics.json"
METRICS = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
print(json.dumps(METRICS, indent=2, sort_keys=True))

## 20. Contact sheet and animation preview

In [ ]:
from IPython.display import Image as DisplayImage, display
for preview_name in ("display_contact_sheet.png", "preview.gif"):
    preview_path = RUN_DIRECTORY / "previews" / preview_name
    if preview_path.is_file():
        display(DisplayImage(filename=str(preview_path)))

## 21. Archive construction

In [ ]:
from flowmorph_klein.packaging import validate_archive
ARCHIVE_REPORT = runner.archive_report
assert ARCHIVE_REPORT is not None and Path(ARCHIVE_REPORT.path).is_file()
ARCHIVE_MEMBERS = validate_archive(ARCHIVE_REPORT.path)
print({"path": str(ARCHIVE_REPORT.path), "size_bytes": ARCHIVE_REPORT.size_bytes, "sha256": ARCHIVE_REPORT.sha256, "members": len(ARCHIVE_MEMBERS)})

## 22. Optional Drive persistence

In [ ]:
if COPY_ARCHIVE_TO_DRIVE:
    from flowmorph_klein.colab_io import stage_drive_inputs, verify_file_checksum
    drive_artifacts = Path(DRIVE_ROOT) / "artifacts"
    copied = stage_drive_inputs(ARCHIVE_REPORT.path, drive_artifacts, overwrite=True)
    copied_archive = copied[0].destination
    verify_file_checksum(copied_archive, ARCHIVE_REPORT.sha256)
    run_manifest = RUN_DIRECTORY / "run_manifest.json"
    if run_manifest.is_file():
        stage_drive_inputs(run_manifest, Path(DRIVE_ROOT) / "manifests", overwrite=True)
    print("Drive copy verified:", copied_archive)
else:
    print("Drive persistence disabled.")

## 23. Final download

In [ ]:
archive_path = Path(ARCHIVE_REPORT.path).resolve()
print("Final archive path:", archive_path)
print("Archive size (bytes):", archive_path.stat().st_size)
print("Archive SHA-256:", ARCHIVE_REPORT.sha256)
print("Reminder: /content is temporary; download or persist this archive now.")
if DOWNLOAD_ARCHIVE:
    try:
        from google.colab import files
    except ImportError:
        print("Colab download helper unavailable; retrieve the plain path shown above.")
    else:
        files.download(str(archive_path))